# Chapter 12: which fix pays first

Does the sums in chapter 12: expected saving, payback and NPV over three years at Kestrel's 12 percent cost of capital. Change a probability in `CHANGES` and the ranking reruns. The last cell draws the effort map.

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fixes = pd.read_csv("../data/fixes.csv").set_index("fix")
RATE, YEARS = 0.12, 3


def annuity(rate, years):
    return sum(1 / (1 + rate) ** t for t in range(1, years + 1))


def rank(f, rate=RATE, value_col="value_usd"):
    out = f.copy()
    out["expected_a_year"] = out[value_col] * out.probability
    out["payback_months"] = out.cost_usd / (out.expected_a_year / 12)
    out["npv_3_years"] = out.expected_a_year * annuity(rate, YEARS) - out.cost_usd
    return out


r = rank(fixes)
print(f"Annuity factor at 12 percent over 3 years: {annuity(RATE, YEARS):.4f}")
r[["name", "value_usd", "probability", "cost_usd", "expected_a_year", "payback_months", "npv_3_years"]].round(1)

In [ ]:
def orders(r):
    return {
        "By value": "".join(r.sort_values("value_usd", ascending=False).index),
        "By payback, shortest first": "".join(r.sort_values("payback_months").index),
        "By risk, safest first": "".join(r.sort_values("probability", ascending=False).index),
        "By NPV": "".join(r.sort_values("npv_3_years", ascending=False).index),
    }


for k, v in orders(r).items():
    print(f"{k}: {', '.join(v)}")
print("Pass the gates this quarter:", ", ".join(r[r.this_quarter == "Yes"].index))

## Fix A in three currencies, at the upper bound and under stress (chapter 12.2)

In [ ]:
a = r.loc["A"]
for label, k in [("Cost", "cost_usd"), ("Expected saving a year", "expected_a_year"), ("NPV over 3 years", "npv_3_years")]:
    print(f"{label}: USD {a[k]:,.0f}, INR {round(a[k]) * 90:,.0f}, EUR {round(a[k]) / 1.1:,.0f}")
ub = rank(fixes, value_col="upper_bound_value_usd").loc["A"]
print(f"Upper bound: expected {ub.expected_a_year:,.0f}, payback {ub.payback_months:.1f} months, NPV {ub.npv_3_years:,.0f}")
half = rank(fixes.assign(probability=fixes.probability.where(fixes.index != "A", 0.30))).loc["A"]
print(f"Probability 0.30: NPV {half.npv_3_years:,.0f} (down {100 * (1 - half.npv_3_years / a.npv_3_years):.0f} percent)")
dear = rank(fixes, rate=0.20).loc["A"]
print(f"Rate 20 percent: NPV {dear.npv_3_years:,.0f} (down {100 * (1 - dear.npv_3_years / a.npv_3_years):.0f} percent)")

## Change a probability and rerun

In [ ]:
CHANGES = {"E": 0.20}  # try your own: {"A": 0.8, "F": 0.5}
f2 = fixes.copy()
for k, p in CHANGES.items():
    f2.loc[k, "probability"] = p
for k, v in orders(rank(f2)).items():
    print(f"{k}: {', '.join(v)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for k, row in fixes.iterrows():
    ax.scatter(row.cost_usd, row.value_usd, s=600 * row.probability, color="#c27c0e" if k == "E" else "#2f7f72", alpha=0.85, edgecolor="#0b4f45")
    ax.annotate(f"{k} {row['name']}, {row.probability:.2f}", (row.cost_usd, row.value_usd), xytext=(10, 0), textcoords="offset points", va="center")
ax.set_xlabel("Effort: one-off cost, USD")
ax.set_ylabel("Value a year, USD")
ax.set_xlim(-2000, 50000)
ax.set_title("Figure 12.2: value against effort; dot size is probability (synthetic)")
plt.tight_layout()
plt.show()